# Altair — Declarative Visualization with Grammar of Graphics

## What is Altair?

Altair is a **declarative** Python visualization library based on the **Grammar of Graphics** — a systematic way of describing visualizations as a mapping from data to visual properties (channels). It compiles to Vega-Lite (a JSON specification) that renders as interactive SVG in the browser.

**Real-world analogy**: Most visualization libraries are like giving someone step-by-step cooking instructions. Altair is like writing a recipe description — you describe *what* the dish should be (ingredients → channels → visual properties), and Altair figures out *how* to render it. This makes Altair code extremely readable and concise.

## What is "Grammar of Graphics"?

The Grammar of Graphics (Leland Wilkinson, 1999) says: **a plot is a mapping from data variables to aesthetic channels** (x, y, color, size, shape, opacity). This idea inspired:
- R's `ggplot2` (most popular R viz library)
- Python's `Altair` (via Vega-Lite)
- Plotly Express (partially)

Once you understand this grammar, you can build any chart by just specifying:
1. **Data** — what DataFrame/dataset?
2. **Mark** — what shape? (point, line, bar, area, text, rect...)
3. **Encoding** — which data column maps to which visual channel? (x, y, color, size...)

## Why Learn Altair?

- **Concise**: Complex multi-layered charts in a few lines
- **Composable**: Layer, facet, concatenate charts like building blocks
- **Interactive**: Built-in selections, tooltips, cross-filtering
- **Shareable**: Outputs pure JSON/HTML — no Python needed to view
- **Learning value**: Understanding Grammar of Graphics makes you better with ALL viz tools

## Prerequisites
- Python basics
- Pandas DataFrames

## Table of Contents
1. Installation & Setup
2. The Grammar: Data + Mark + Encoding
3. Core Mark Types
4. Encoding Channels (x, y, color, size, shape, opacity, tooltip)
5. Data Transformations in Altair
6. Layering & Combining Charts
7. Faceting (Small Multiples)
8. Interactivity: Selections & Cross-filtering
9. Themes & Color Schemes
10. Saving & Exporting
11. Common Pitfalls
12. Mini Project: Interactive EDA Dashboard
13. Interview Q&A
14. Resources

---

**Official Docs**: https://altair-viz.github.io/  
**Gallery**: https://altair-viz.github.io/gallery/  
**Vega-Lite Spec** (what Altair compiles to): https://vega.github.io/vega-lite/  
**YouTube**: https://www.youtube.com/watch?v=aRxahWy-ul8  
**Original Grammar of Graphics** book: https://link.springer.com/book/10.1007/0-387-28695-0

## 1. Installation & Setup

```bash
pip install altair pandas numpy
# For Vega datasets:
pip install vega_datasets
```

In Jupyter, Altair renders inline automatically. The output is an interactive SVG — you can hover, zoom, and click without any extra setup.

In [ ]:
import altair as alt
import pandas as pd
import numpy as np
from vega_datasets import data as vega_data  # Built-in datasets

# In Jupyter, Altair renders inline. No extra setup needed.
print(f"Altair version: {alt.__version__}")

# Altair has a row limit of 5000 by default to prevent browser slowdowns
# For larger datasets:
alt.data_transformers.enable('default', max_rows=10000)

# Available Vega datasets
print("\nSome available datasets:")
from vega_datasets import local_data
print(list(local_data.list_datasets())[:10])

---
## 2. The Grammar: Data + Mark + Encoding

Every Altair chart follows this pattern:

```python
alt.Chart(data)          # 1. What data?
   .mark_point()         # 2. What shape (mark)?
   .encode(              # 3. What maps to what channel?
       x='column_name',
       y='column_name',
       color='column_name'
   )
```

This is the **entire mental model**. Everything else is variations on these three components.

### Data Type Shortcodes
Altair uses shortcodes in channel strings to specify the data type:

| Shortcode | Type | Meaning | Example |
|-----------|------|---------|--------|
| `:Q` | Quantitative | Continuous number | `'price:Q'` |
| `:O` | Ordinal | Ordered categories | `'rating:O'` |
| `:N` | Nominal | Unordered categories | `'color:N'` |
| `:T` | Temporal | Date/time | `'date:T'` |

Altair often infers the type, but being explicit prevents surprises.

In [ ]:
# Simple example: Iris dataset
iris = vega_data.iris()
print("Iris dataset head:")
print(iris.head())
print("\nDtypes:", iris.dtypes.to_dict())

# The simplest possible Altair chart: 3 lines!
chart = alt.Chart(iris).mark_point().encode(
    x='sepalLength:Q',
    y='sepalWidth:Q',
    color='species:N'  # N = Nominal (unordered categories)
)
chart

---
## 3. Core Mark Types

| Mark Method | Shape | Use For |
|------------|-------|---------|
| `mark_point()` | Points | Scatter plots |
| `mark_circle()` | Filled circles | Scatter (cleaner) |
| `mark_line()` | Line | Time series, trends |
| `mark_bar()` | Bars | Bar charts, histograms |
| `mark_area()` | Filled area | Area charts, streams |
| `mark_text()` | Text labels | Annotations, labels |
| `mark_rect()` | Rectangle | Heatmaps |
| `mark_tick()` | Thin tick | Strip plots |
| `mark_rule()` | Full-width line | Reference lines |
| `mark_boxplot()` | Box + whisker | Box plots (in one!) |

In [ ]:
# Load the cars dataset
cars = vega_data.cars()
print("Cars dataset columns:", list(cars.columns))

# ── 4 charts with different marks ────────────────────────────────────────────
base = alt.Chart(cars).encode(
    x='Horsepower:Q',
    y='Miles_per_Gallon:Q',
    color='Origin:N'
)

# mark_circle: filled circles (cleaner than mark_point)
chart_circle = base.mark_circle(opacity=0.6).properties(
    title='Horsepower vs MPG (circles)', width=300, height=220
)

# mark_tick: thin tick marks (like stripplot)
chart_tick = alt.Chart(cars).mark_tick(thickness=2).encode(
    x='Horsepower:Q',
    y='Origin:N',
    color='Origin:N'
).properties(title='Horsepower by Origin (ticks)', width=300, height=220)

# mark_bar: histogram of MPG
chart_bar = alt.Chart(cars).mark_bar().encode(
    x=alt.X('Miles_per_Gallon:Q', bin=alt.Bin(maxbins=20)),
    y='count():Q',
    color='Origin:N'
).properties(title='MPG Distribution (histogram)', width=300, height=220)

# mark_boxplot: box plot (built-in!)
chart_box = alt.Chart(cars).mark_boxplot(extent='min-max').encode(
    x='Origin:N',
    y='Miles_per_Gallon:Q',
    color='Origin:N'
).properties(title='MPG by Origin (box)', width=300, height=220)

# Concatenate horizontally
((chart_circle | chart_tick) & (chart_bar | chart_box)).properties(
    title=alt.TitleParams('Core Mark Types in Altair', fontSize=15)
)

---
## 4. Encoding Channels (Advanced)

Channels are more than just `x` and `y`. The full set:

| Channel | Visual Property |
|---------|----------------|
| `x`, `y` | Position |
| `color` | Fill/stroke color |
| `size` | Point/bar size |
| `shape` | Marker shape (only for mark_point) |
| `opacity` | Transparency |
| `tooltip` | Hover tooltip content |
| `text` | Text label (for mark_text) |
| `detail` | Grouping without visual change |
| `column` / `row` | Facet (small multiples) |
| `order` | Line ordering / stacking order |

For each channel, you can use the full `alt.X(...)` syntax for fine control:
```python
x=alt.X('column:Q', 
          scale=alt.Scale(type='log'),       # Log scale
          axis=alt.Axis(title='My Label'),   # Custom axis
          sort='descending')                 # Sort order
```

In [ ]:
# ── Using the full alt.X() syntax for control ──────────────────────────────────
gapminder = vega_data.gapminder()
gap_2000  = gapminder[gapminder['year'] == 2000].dropna()

# All 5 encoding channels used!
chart_rich = alt.Chart(gap_2000).mark_circle().encode(
    x=alt.X('fertility:Q',
              scale=alt.Scale(zero=False),
              axis=alt.Axis(title='Fertility Rate')),
    y=alt.Y('life_expect:Q',
              scale=alt.Scale(zero=False),
              axis=alt.Axis(title='Life Expectancy (years)')),
    size=alt.Size('pop:Q',
                   scale=alt.Scale(range=[10, 800]),  # min/max pixel sizes
                   legend=alt.Legend(title='Population')),
    color=alt.Color('cluster:N',
                     scale=alt.Scale(scheme='category10'),
                     legend=alt.Legend(title='Region')),
    tooltip=[
        alt.Tooltip('country:N', title='Country'),
        alt.Tooltip('fertility:Q', title='Fertility', format='.2f'),
        alt.Tooltip('life_expect:Q', title='Life Expectancy', format='.1f'),
        alt.Tooltip('pop:Q', title='Population', format=',d'),
    ]
).properties(
    title='World Development Indicators (2000)',
    width=550, height=350
)

chart_rich

---
## 5. Data Transformations in Altair

Altair can perform transformations **inside the chart specification** (without modifying the DataFrame). This is powerful because the transformations travel with the chart.

| Transform | Purpose |
|-----------|--------|
| `.transform_filter()` | Filter rows |
| `.transform_calculate()` | Compute a new column |
| `.transform_aggregate()` | Group and aggregate |
| `.transform_fold()` | Wide → Long (like pd.melt) |
| `.transform_density()` | KDE estimation |
| `.transform_bin()` | Binning |

In [ ]:
# ── transform_filter: show only USA cars ──────────────────────────────────────
chart_usa = alt.Chart(cars).mark_circle(color='steelblue', opacity=0.7).encode(
    x='Horsepower:Q',
    y='Miles_per_Gallon:Q',
    tooltip=['Name:N', 'Horsepower:Q', 'Miles_per_Gallon:Q', 'Year:O']
).transform_filter(
    alt.datum.Origin == 'USA'  # Filter to only USA cars
).properties(title='USA Cars Only', width=350, height=250)

# ── transform_calculate: add a computed column ────────────────────────────────
chart_calc = alt.Chart(cars).mark_circle(opacity=0.7).encode(
    x='Horsepower:Q',
    y='power_to_weight:Q',  # This is computed below!
    color='Origin:N',
    tooltip=['Name:N', 'Horsepower:Q', 'Weight_in_lbs:Q']
).transform_calculate(
    power_to_weight='datum.Horsepower / datum.Weight_in_lbs * 1000'  # New column
).properties(title='Horsepower vs Power/Weight Ratio', width=350, height=250)

chart_usa | chart_calc

In [ ]:
# ── transform_aggregate: group by origin, compute avg HP and count ────────────
chart_agg = alt.Chart(cars).mark_bar().encode(
    x=alt.X('mean_hp:Q', title='Avg Horsepower'),
    y=alt.Y('Origin:N', sort='-x'),
    color='Origin:N',
    tooltip=['Origin:N', 
              alt.Tooltip('mean_hp:Q', title='Avg HP', format='.1f'),
              alt.Tooltip('count:Q', title='# Cars')]
).transform_aggregate(
    mean_hp='mean(Horsepower)',
    count='count()',
    groupby=['Origin']
).properties(title='Avg Horsepower by Origin', width=400, height=180)

# ── transform_fold: compare multiple metrics side by side (like pd.melt) ──────
# Fold: takes wide format → long format inside the chart
chart_fold = alt.Chart(cars).mark_line().encode(
    x='Year:O',
    y=alt.Y('mean(value):Q', title='Average Value'),
    color='metric:N'
).transform_fold(
    ['Horsepower', 'Miles_per_Gallon', 'Acceleration'],  # Columns to fold
    as_=['metric', 'value']  # New column names
).properties(title='Trends Over Time (folded)', width=400, height=180)

chart_agg | chart_fold

---
## 6. Layering & Combining Charts

Altair's composition operators are one of its best features:

| Operator | Method | Effect |
|----------|--------|--------|
| `+` | `layer()` | Stack charts on same axes (like adding matplotlib traces) |
| `|` | `hconcat()` | Place charts side by side |
| `&` | `vconcat()` | Stack charts vertically |

**Layering** is especially useful for: line + points, scatter + regression line, bar + text labels.

In [ ]:
# ── Layered: Line + Points + Text ─────────────────────────────────────────────
monthly_df = pd.DataFrame({
    'month': ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'],
    'sales': [185, 192, 205, 198, 220, 235, 285, 260, 248, 265, 290, 320],
    'target':[200, 205, 210, 215, 225, 235, 250, 260, 265, 270, 280, 300]
})
monthly_df['month_order'] = range(12)
monthly_df['above_target'] = monthly_df['sales'] >= monthly_df['target']

base_monthly = alt.Chart(monthly_df)

# Line for actual sales
line_sales = base_monthly.mark_line(strokeWidth=3).encode(
    x=alt.X('month:O', sort=list(monthly_df['month']), axis=alt.Axis(labelAngle=0)),
    y=alt.Y('sales:Q', scale=alt.Scale(domain=[150, 340])),
    tooltip=['month:O', 'sales:Q']
).properties(width=500, height=260)

# Colored dots (green if ≥ target, red if below)
dots = base_monthly.mark_circle(size=120).encode(
    x=alt.X('month:O', sort=list(monthly_df['month'])),
    y='sales:Q',
    color=alt.Color('above_target:N',
                     scale=alt.Scale(domain=[True, False], range=['#2ecc71', '#e74c3c']),
                     legend=alt.Legend(title='Above Target?'))
)

# Dashed line for target
line_target = base_monthly.mark_line(strokeDash=[5, 5], color='gray', strokeWidth=2).encode(
    x=alt.X('month:O', sort=list(monthly_df['month'])),
    y='target:Q'
)

# Text labels for sales
text_labels = base_monthly.mark_text(dy=-12, fontSize=9, color='#2c3e50').encode(
    x=alt.X('month:O', sort=list(monthly_df['month'])),
    y='sales:Q',
    text=alt.Text('sales:Q', format='d')
)

# Layer all 4 together!
(line_sales + dots + line_target + text_labels).properties(
    title=alt.TitleParams('Monthly Sales vs Target', fontSize=14)
)

---
## 7. Faceting (Small Multiples)

Faceting creates **multiple small charts** — one per value of a categorical variable. This is one of the most powerful EDA techniques: instead of overlapping many groups in one chart, you give each group its own panel.

**Analogy**: Instead of showing all students' grades on one chart (tangled mess), you show one chart per student — patterns become immediately visible.

In [ ]:
# ── Facet: Distribution by origin ─────────────────────────────────────────────
chart_facet = alt.Chart(cars).mark_bar().encode(
    x=alt.X('Miles_per_Gallon:Q', bin=alt.Bin(maxbins=15)),
    y='count():Q',
    color='Origin:N'
).properties(
    width=180, height=140
).facet(
    facet=alt.Facet('Origin:N', header=alt.Header(labelFontSize=12)),
    columns=3  # Arrange in rows of 3
).properties(
    title='MPG Distribution by Country of Origin'
)

chart_facet

In [ ]:
# ── col= and row= in encode() for 2D facets ──────────────────────────────────
# Create a categorical variable
cars['mpg_class'] = pd.cut(cars['Miles_per_Gallon'], bins=[0, 20, 30, 100],
                            labels=['Low (<20)', 'Medium (20-30)', 'High (>30)'])

chart_2d = alt.Chart(cars.dropna(subset=['Miles_per_Gallon', 'mpg_class'])).mark_circle(
    opacity=0.6
).encode(
    x='Horsepower:Q',
    y='Weight_in_lbs:Q',
    color='Origin:N',
    tooltip=['Name:N', 'Horsepower:Q', 'Weight_in_lbs:Q', 'Miles_per_Gallon:Q']
).properties(
    width=200, height=160
).facet(
    row=alt.Row('mpg_class:N', sort=['Low (<20)', 'Medium (20-30)', 'High (>30)']),
    column='Origin:N'
).properties(
    title='HP vs Weight: Faceted by MPG Class and Origin'
)

chart_2d

---
## 8. Interactivity: Selections & Cross-Filtering

Altair's interaction model is selection-based:
1. **Define a selection** — a named selection that captures user input
2. **Bind the selection** to a chart element (click, interval, multi-select)
3. **Filter or modify** other parts of the chart based on the selection

| Selection Type | User Action | Use Case |
|---------------|------------|----------|
| `selection_single` | Click one point | Highlight one category |
| `selection_multi` | Shift+click | Highlight multiple |
| `selection_interval` | Drag rectangle | Brush/zoom |

The amazing part: **one selection can filter multiple charts simultaneously** (cross-filtering).

In [ ]:
# ── Cross-filter: selecting in one chart filters others ────────────────────────
# Click on a bar to filter the scatter plot!

brush = alt.selection_interval()  # Drag to select a region
click = alt.selection_point(fields=['Origin'])  # Click to select an origin

# Scatter plot: affected by brush selection
scatter = alt.Chart(cars).mark_circle().encode(
    x='Horsepower:Q',
    y='Miles_per_Gallon:Q',
    color=alt.condition(
        brush,                         # IF selected by brush...
        'Origin:N',                    # THEN use color by Origin
        alt.value('lightgray')         # ELSE gray out
    ),
    opacity=alt.condition(brush, alt.value(0.8), alt.value(0.1)),
    tooltip=['Name:N', 'Origin:N', 'Horsepower:Q', 'Miles_per_Gallon:Q']
).add_params(brush).properties(width=400, height=280, title='Drag to select')

# Bar chart: affected by brush, can also be clicked
bars = alt.Chart(cars).mark_bar().encode(
    x=alt.X('count():Q', title='# Cars in Selection'),
    y='Origin:N',
    color=alt.Color('Origin:N'),
    opacity=alt.condition(click, alt.value(1), alt.value(0.4))
).transform_filter(
    brush  # FILTER to only brushed points!
).add_params(click).properties(width=250, height=280, title='Click to highlight')

(scatter | bars).properties(
    title=alt.TitleParams(
        'Cross-Filter: Drag scatter to filter bar | Click bar to highlight',
        fontSize=12, anchor='start'
    )
)

In [ ]:
# ── Dropdown selector bound to chart ─────────────────────────────────────────
# User picks an origin from a dropdown → chart filters
origins = sorted(cars['Origin'].unique())
origin_dropdown = alt.binding_select(options=origins + ['All'], name='Origin: ')
selection_drop  = alt.selection_point(fields=['Origin'], bind=origin_dropdown, value='All')

chart_dropdown = alt.Chart(cars).mark_circle(opacity=0.7, size=80).encode(
    x='Horsepower:Q',
    y='Miles_per_Gallon:Q',
    color='Origin:N',
    tooltip=['Name:N', 'Origin:N', 'Horsepower:Q', 'Miles_per_Gallon:Q', 'Year:O']
).add_params(
    selection_drop
).transform_filter(
    # Show all if 'All', else filter to selected
    (alt.datum.Origin == selection_drop.Origin) | (selection_drop.Origin == 'All')
).properties(
    title='Select Origin from Dropdown',
    width=450, height=280
)

chart_dropdown

---
## 9. Themes & Color Schemes

Altair supports named color schemes from Vega (https://vega.github.io/vega/docs/schemes/):
- **Sequential**: `'viridis'`, `'plasma'`, `'blues'`, `'greens'`, `'oranges'`
- **Diverging**: `'redblue'`, `'redyellowgreen'`, `'spectral'`
- **Categorical**: `'category10'`, `'category20'`, `'set1'`, `'set2'`, `'dark2'`
- **Cyclical**: `'rainbow'`, `'sinebow'`

Themes change the overall look: `alt.themes.enable('ggplot2')`, `'fivethirtyeight'`, `'vox'`, `'dark'`

In [ ]:
# ── Continuous color scale (sequential) ───────────────────────────────────────
chart_seq = alt.Chart(cars).mark_circle(size=80).encode(
    x='Horsepower:Q',
    y='Miles_per_Gallon:Q',
    color=alt.Color('Acceleration:Q',
                     scale=alt.Scale(scheme='viridis'),
                     legend=alt.Legend(title='Acceleration')),
    tooltip=['Name:N', 'Horsepower:Q', 'Miles_per_Gallon:Q', 'Acceleration:Q']
).properties(title='Sequential colormap: viridis', width=350, height=220)

# ── Diverging color scale (for +/- data) ──────────────────────────────────────
# Heatmap with correlation data
numeric_cols = ['Horsepower', 'Miles_per_Gallon', 'Acceleration', 'Weight_in_lbs', 'Cylinders']
corr = cars[numeric_cols].dropna().corr().reset_index().melt('index')
corr.columns = ['var1', 'var2', 'correlation']

chart_heatmap = alt.Chart(corr).mark_rect().encode(
    x='var1:O',
    y='var2:O',
    color=alt.Color('correlation:Q',
                     scale=alt.Scale(scheme='redblue', domain=[-1, 1]),
                     legend=alt.Legend(title='Correlation')),
    tooltip=['var1:N', 'var2:N', alt.Tooltip('correlation:Q', format='.3f')]
).properties(title='Diverging colormap: redblue', width=250, height=250)

# Annotation layer for correlation values
text_layer = alt.Chart(corr).mark_text(fontSize=9).encode(
    x='var1:O',
    y='var2:O',
    text=alt.Text('correlation:Q', format='.2f'),
    color=alt.condition(
        abs(alt.datum.correlation) > 0.5,
        alt.value('white'),
        alt.value('black')
    )
)

(chart_seq | (chart_heatmap + text_layer)).properties(
    title=alt.TitleParams('Color Schemes in Altair', fontSize=13)
)

---
## 10. Saving & Exporting

In [ ]:
import os

# Create a chart to save
chart_save = alt.Chart(iris).mark_circle().encode(
    x='sepalLength:Q', y='sepalWidth:Q', color='species:N'
).properties(title='Export Example', width=400, height=300)

# ── Save as HTML (interactive) ────────────────────────────────────────────────
chart_save.save('/tmp/altair_chart.html')
print(f"HTML saved: {os.path.getsize('/tmp/altair_chart.html'):,} bytes")

# ── Save as JSON (Vega-Lite spec) ─────────────────────────────────────────────
chart_save.save('/tmp/altair_chart.json')
print(f"JSON saved: {os.path.getsize('/tmp/altair_chart.json'):,} bytes")

# ── Show the Vega-Lite JSON spec ──────────────────────────────────────────────
import json
spec = chart_save.to_dict()
print(f"\nVega-Lite spec (Altair compiles to this JSON):")
print(json.dumps(spec, indent=2)[:800] + '...')

# ── Static PNG/SVG (requires vl-convert or altair_saver) ─────────────────────
try:
    chart_save.save('/tmp/altair_chart.png')
    print(f"\nPNG saved: {os.path.getsize('/tmp/altair_chart.png'):,} bytes")
except Exception as e:
    print(f"\nStatic PNG requires: pip install vl-convert-python")
    print(f"Then: import vl_convert as vlc (happens automatically)")

---
## 11. Common Pitfalls

| Pitfall | Problem | Fix |
|---------|---------|-----|
| Row limit (default 5000) | `MaxRowsError` with large datasets | `alt.data_transformers.enable('default', max_rows=None)` or aggregate first |
| Wrong data type shortcode | Temporal column treated as ordinal | Explicitly use `:T` for dates |
| `sort` ignored | Alphabetical sort instead of data order | Use `sort=alt.EncodingSortField(field='col', order='descending')` |
| Chart not showing in Jupyter | Renderer issue | `alt.renderers.enable('default')` |
| Composition (`|`, `&`, `+`) order | Complex nesting creates wrong layout | Use parentheses liberally: `(a | b) & c` |
| Cross-filter not working | Selection names conflict | Give selections unique `name='sel_name'` |
| `condition()` for color | TypeError if field and value types differ | Use `alt.value('gray')` for constant values |

---
## 12. Mini Project: Interactive EDA Dashboard with Cross-Filtering

**Scenario**: You're a data analyst for a car manufacturer. Build a dashboard that allows:
1. Brush across the scatter plot to filter → bar chart updates
2. Click a bar to filter → distribution updates
3. Everything links together with a single selection

**Goal**: Demonstrate Altair's composability and cross-filtering in a meaningful analytical context.

In [ ]:
import altair as alt
import pandas as pd
import numpy as np
from vega_datasets import data as vega_data

# ── Load and prepare data ──────────────────────────────────────────────────────
cars = vega_data.cars()
cars = cars.dropna()

# ── Define shared selections ────────────────────────────────────────────────────
brush   = alt.selection_interval(resolve='global', name='brush')
clicked = alt.selection_point(fields=['Origin'], name='clicked')

# ── Color encoding: gray out non-selected ─────────────────────────────────────
color_brushed = alt.condition(
    brush,
    alt.Color('Origin:N', scale=alt.Scale(scheme='set1')),
    alt.value('lightgray')
)

# ── Chart 1: Scatter — HP vs MPG, brush to filter ─────────────────────────────
scatter = alt.Chart(cars).mark_circle(size=70).encode(
    x=alt.X('Horsepower:Q', title='Horsepower', scale=alt.Scale(zero=False)),
    y=alt.Y('Miles_per_Gallon:Q', title='Miles per Gallon', scale=alt.Scale(zero=False)),
    color=color_brushed,
    opacity=alt.condition(brush, alt.value(0.8), alt.value(0.1)),
    tooltip=[
        alt.Tooltip('Name:N'),
        alt.Tooltip('Origin:N'),
        alt.Tooltip('Horsepower:Q', format='.0f'),
        alt.Tooltip('Miles_per_Gallon:Q', format='.1f'),
        alt.Tooltip('Year:O')
    ]
).add_params(brush).properties(
    width=380, height=280,
    title='Drag to select a region →'
)

# ── Chart 2: Bar — # cars by Origin (filtered by brush) ──────────────────────
bars_origin = alt.Chart(cars).mark_bar().encode(
    x=alt.X('count():Q', title='Count in Selection'),
    y=alt.Y('Origin:N', sort='-x'),
    color=alt.condition(
        clicked,
        alt.Color('Origin:N', scale=alt.Scale(scheme='set1'), legend=None),
        alt.value('lightgray')
    ),
    tooltip=['Origin:N', 'count():Q']
).transform_filter(
    brush  # Only count brushed points
).add_params(clicked).properties(
    width=260, height=280,
    title='← Click to highlight an origin'
)

# ── Chart 3: Distribution — Weight by Year (filtered by brush + clicked) ──────
dist_weight = alt.Chart(cars).mark_bar(opacity=0.7).encode(
    x=alt.X('Weight_in_lbs:Q', bin=alt.Bin(maxbins=20), title='Weight (lbs)'),
    y=alt.Y('count():Q', stack=None, title='Count'),
    color=alt.Color('Origin:N', scale=alt.Scale(scheme='set1')),
).transform_filter(
    brush
).transform_filter(
    clicked
).properties(
    width=640, height=150,
    title='Weight Distribution of Selected Cars'
)

# ── Chart 4: Time trend (filtered by brush + clicked) ─────────────────────────
trend_year = alt.Chart(cars).mark_line(point=True).encode(
    x=alt.X('Year:O'),
    y=alt.Y('mean(Miles_per_Gallon):Q', title='Avg MPG'),
    color=alt.Color('Origin:N', scale=alt.Scale(scheme='set1'))
).transform_filter(
    brush
).transform_filter(
    clicked
).properties(
    width=640, height=150,
    title='Avg MPG Over Time (for Selection)'
)

# ── Compose dashboard ─────────────────────────────────────────────────────────
dashboard = (
    (scatter | bars_origin) &
    dist_weight &
    trend_year
).properties(
    title=alt.TitleParams(
        'Car Dataset: Interactive Cross-Filter Dashboard',
        fontSize=16, anchor='middle'
    )
).configure_view(
    stroke=None  # Remove border
).configure_axis(
    labelFontSize=10, titleFontSize=11
)

dashboard.save('/tmp/altair_dashboard.html')
print("Dashboard saved to /tmp/altair_dashboard.html")
dashboard

---
## 13. Interview Q&A

**Q1: What is the Grammar of Graphics and how does Altair implement it?**  
**A**: The Grammar of Graphics (Wilkinson, 1999) treats visualizations as a structured mapping from data to visual properties (x, y, color, size, shape). Altair implements it through: `alt.Chart(data)` (data), `.mark_*()` (geometric primitive), `.encode()` (data-to-channel mapping), and optional `.transform_*()` (data transformation). This structure means any chart type can be built from the same 3 components.

---
**Q2: What is the difference between a selection and a filter in Altair?**  
**A**: A `selection` captures user input (click, brush) and stores it. A `transform_filter` uses a selection or predicate to show only matching rows. `condition()` uses a selection to toggle visual properties (color, opacity) without hiding rows. Together, they enable cross-filtering: `transform_filter(selection)` makes another chart show only the selected data.

---
**Q3: What does Altair compile to, and why is that significant?**  
**A**: Altair compiles Python code to **Vega-Lite JSON** specifications. This means: (1) charts are language-agnostic — share the JSON and anyone can render it, (2) the spec can be saved and reloaded, (3) the same spec renders in Python, JavaScript, R, Rust, or any Vega-Lite implementation, (4) charts are embeddable in static HTML without a Python server.

---
**Q4: How do you handle datasets larger than Altair's 5000-row limit?**  
**A**: Options: (1) Pre-aggregate in Pandas: `df.groupby(...).agg(...)`, (2) `alt.data_transformers.enable('default', max_rows=None)` — removes the limit but may slow down the browser, (3) `alt.data_transformers.enable('vegafusion')` — processes data server-side with VegaFusion, (4) Use `transform_aggregate` inside Altair to aggregate before rendering.

---
**Q5: How does Altair's `|`, `&`, and `+` composition differ?**  
**A**: `chart1 | chart2` → horizontal concatenation (side by side). `chart1 & chart2` → vertical concatenation (stacked). `chart1 + chart2` → layering (overlaid on same axes — like adding multiple traces). Combine them: `(a | b) & (c + d)` creates a 2-row layout where the bottom row has two charts layered.

---
**Q6: When would you choose Altair over Plotly?**  
**A**: Choose Altair when: you value concise, readable code that maps naturally to the Grammar of Graphics; you need sophisticated cross-filtering with minimal code; you want to generate Vega-Lite specs for JavaScript consumption; you're doing academic/research visualization where code readability matters. Choose Plotly when: you need 3D charts, geographic maps, or you're building Dash web apps with Python server callbacks.

---
## 14. Resources

### Official
- **Altair Documentation**: https://altair-viz.github.io/
- **Altair Gallery**: https://altair-viz.github.io/gallery/
- **Vega-Lite Spec** (what Altair compiles to): https://vega.github.io/vega-lite/
- **Vega Color Schemes**: https://vega.github.io/vega/docs/schemes/

### YouTube
- **Altair Tutorial by Jodie Burchell**: https://www.youtube.com/watch?v=aRxahWy-ul8
- **Python Viz with Altair (SciPy talk)**: https://www.youtube.com/watch?v=ms29ZPUKxbU

### Books & Papers
- **Altair Paper**: VanderPlas et al., "Altair: Interactive Statistical Visualizations for Python", JOSS 2018: https://joss.theoj.org/papers/10.21105/joss.01057
- **The Grammar of Graphics** (Wilkinson, 2005): https://link.springer.com/book/10.1007/0-387-28695-0
- **Vega-Lite Paper**: Satyanarayan et al., "Vega-Lite: A Grammar of Interactive Graphics", IEEE VIS 2017: https://idl.cs.washington.edu/papers/vega-lite

---
## Summary & What's Next

### What You Learned
| Concept | Key Point |
|---------|----------|
| Grammar of Graphics | Data + Mark + Encoding = any chart |
| Data type codes | `:Q` (quantitative), `:N` (nominal), `:O` (ordinal), `:T` (temporal) |
| Mark types | point, circle, line, bar, area, text, rect, tick, rule, boxplot |
| Encoding channels | x, y, color, size, shape, opacity, tooltip, detail |
| Transforms | filter, calculate, aggregate, fold, density, bin |
| Composition | `+` (layer), `|` (horizontal), `&` (vertical) |
| Faceting | `.facet()` or `column=`, `row=` in encoding |
| Interactivity | `selection_interval`, `selection_point`, `condition()`, `transform_filter()` |
| Export | `.save('chart.html')`, `.save('chart.json')`, `.to_dict()` |

### The Visualization Landscape — Which to Use?
| Library | Best For |
|---------|----------|
| **Matplotlib** | Publication-quality static figures, custom layouts |
| **Seaborn** | Quick statistical EDA, beautiful defaults |
| **Plotly** | Interactive dashboards, 3D, maps |
| **Bokeh** | Streaming data, Python server callbacks, linked brushing |
| **Altair** | Concise Grammar-of-Graphics, cross-filtering, Vega-Lite output |

### What's Next?
- **Next Phase**: Classical Machine Learning (Scikit-Learn, XGBoost, LightGBM, CatBoost)
- **Practice**: Recreate a chart from the Altair gallery using your own dataset
- **Challenge**: Build a full cross-filter dashboard for a Kaggle dataset using only Altair

> **Key insight**: Once you understand the Grammar of Graphics through Altair, you'll find yourself thinking more clearly about visualization in general — "what data maps to what channel?" is the right question for any chart in any tool.